# Notebook 04 — Quantitative Evaluation

**Phase 5 | Metrics: CLIPScore · DINOv2 · LPIPS**

This notebook:
1. Runs all three evaluation metrics across cells A–D.
2. Produces bar charts and a summary table for the slide deck.
3. Displays the best and worst outputs per cell.
4. Contains the findings write-up template (fill in after reviewing results).

> **Prerequisite:** notebooks 02 and 03 must have been run successfully (80 + 80 images in `outputs/A–D/`).

## 0 — Colab Bootstrap

In [ ]:
import os, sys

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    REPO_URL = 'https://github.com/<YOUR_USERNAME>/stable-diffusion.git'  # ← update
    REPO_DIR = '/content/stable-diffusion'

    if not os.path.isdir(REPO_DIR):
        !git clone $REPO_URL $REPO_DIR
    else:
        !git -C $REPO_DIR pull

    %cd $REPO_DIR
    !pip install -q -r requirements.txt
else:
    root = os.path.abspath(os.path.join(os.getcwd(), '..'))
    if os.path.basename(root) == 'stable-diffusion':
        os.chdir(root)
    print(f'Working dir: {os.getcwd()}')

## 1 — Imports & Setup

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
from PIL import Image

from src.data.dataset import OxfordPetDataset
from src.evaluation.clip_score import clip_scores_from_dir
from src.evaluation.dino_sim import dino_scores_from_dirs
from src.evaluation.lpips_diversity import lpips_scores_from_dirs
from src.evaluation.report import build_report

OUT_ROOT  = Path('outputs')
EVAL_DIR  = Path('outputs/eval')
CELLS     = ['A', 'B', 'C', 'D']
DATA_ROOT = 'data'

CELL_LABELS = {
    'A': 'A — Naive',
    'B': 'B — Naive+CN',
    'C': 'C — Structured',
    'D': 'D — Struct+CN',
}
CELL_COLORS = {
    'A': '#4c8ed9',
    'B': '#3a6ca8',
    'C': '#4cba88',
    'D': '#2e8a5c',
}

plt.style.use('dark_background')
plt.rcParams.update({'font.family': 'DejaVu Sans', 'axes.facecolor': '#1a1a1a',
                     'figure.facecolor': '#0d0d0d', 'axes.edgecolor': '#444',
                     'axes.labelcolor': '#ccc', 'xtick.color': '#ccc',
                     'ytick.color': '#ccc', 'grid.color': '#333'})

print('Imports OK')

## 2 — Check Output Counts

In [ ]:
for cell in CELLS:
    cell_dir = OUT_ROOT / cell
    if cell_dir.exists():
        pngs = list(cell_dir.glob('*.png'))
        jsons = list(cell_dir.glob('*.json'))
        status = '✓' if len(pngs) == len(jsons) and len(pngs) > 0 else '✗'
        print(f'Cell {cell}: {len(pngs):4d} images, {len(jsons):4d} sidecars  {status}')
    else:
        print(f'Cell {cell}: MISSING — run notebooks 02 and 03 first')

## 3 — Load Dataset (for DINOv2 breed reference images)

In [ ]:
try:
    dataset = OxfordPetDataset(root=DATA_ROOT)
    print(f'Dataset loaded: {len(dataset)} images')
except Exception as e:
    print(f'WARNING: {e}\nBreed identity scores will be NaN. Run download_data.py first.')
    dataset = None

## 4 — Run Full Evaluation

This calls all three metrics and writes `outputs/eval/summary.csv` and `cell_summary.csv`.  
**On T4, expect ~5–10 min** (CLIP + DINOv2 on 160 images, LPIPS pairwise).

In [ ]:
per_image_df, cell_df = build_report(
    output_root=OUT_ROOT,
    eval_dir=EVAL_DIR,
    cells=CELLS,
    dataset=dataset,
)

print(f'rows: {len(per_image_df)}')
per_image_df.head()

## 5 — Cell Summary Table

In [ ]:
# Pretty-print the cell summary
display_cols = [c for c in cell_df.columns if '_mean' in c]
display = cell_df[['cell'] + display_cols].copy()
display.columns = ['Cell'] + [c.replace('_mean', '') for c in display_cols]
display['Prompt']     = display['Cell'].map({'A':'Naive','B':'Naive','C':'Structured','D':'Structured'})
display['ControlNet'] = display['Cell'].map({'A':'No','B':'Yes','C':'No','D':'Yes'})
display.insert(1, 'Prompt', display.pop('Prompt'))
display.insert(2, 'ControlNet', display.pop('ControlNet'))
print(display.to_string(index=False))

## 6 — Bar Charts

One subplot per metric. Designed for copy-paste into a slide.

In [ ]:
metrics = [
    ('clip_score',             'CLIPScore\n(↑ better alignment)',    0.15, 0.45),
    ('breed_identity',         'Breed Identity\n(DINOv2 ↑)',         0.0,  1.0),
    ('condition_consistency',  'Condition Consistency\n(DINOv2 ↑)',  0.0,  1.0),
    ('lpips_diversity',        'LPIPS Diversity\n(↑ more varied)',   0.0,  1.0),
]

fig, axes = plt.subplots(1, len(metrics), figsize=(16, 5))
fig.suptitle('2×2 Ablation — Quantitative Metric Comparison', color='white', fontsize=14)

x = np.arange(len(CELLS))
bar_w = 0.55

for ax, (col, ylabel, ymin, ymax) in zip(axes, metrics):
    mean_col = f'{col}_mean'
    std_col  = f'{col}_std'

    means = []
    stds  = []
    for cell in CELLS:
        row = cell_df[cell_df['cell'] == cell]
        means.append(float(row[mean_col].values[0]) if mean_col in row.columns and not row.empty else 0)
        stds.append(float(row[std_col].values[0])   if std_col  in row.columns and not row.empty else 0)

    bars = ax.bar(
        x, means, width=bar_w,
        color=[CELL_COLORS[c] for c in CELLS],
        edgecolor='#555', linewidth=0.8,
    )
    ax.errorbar(x, means, yerr=stds, fmt='none', color='white',
                capsize=4, elinewidth=1.5)

    # Value labels on bars
    for bar, m in zip(bars, means):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                f'{m:.3f}', ha='center', va='bottom', color='white', fontsize=8)

    ax.set_ylabel(ylabel, fontsize=9)
    ax.set_xticks(x)
    ax.set_xticklabels([CELL_LABELS[c] for c in CELLS], fontsize=8, rotation=20, ha='right')
    ax.set_ylim(ymin, ymax * 1.12)
    ax.grid(axis='y', linewidth=0.5, alpha=0.6)
    ax.axhline(0, color='#555', linewidth=0.8)

plt.tight_layout()
chart_path = EVAL_DIR / 'metrics_bar_chart.png'
plt.savefig(chart_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Chart saved → {chart_path}')

## 7 — Per-Metric Distribution (Box Plots)

In [ ]:
metric_cols = ['clip_score', 'breed_identity', 'condition_consistency', 'lpips_diversity']
metric_cols = [c for c in metric_cols if c in per_image_df.columns]

fig, axes = plt.subplots(1, len(metric_cols), figsize=(16, 5))
fig.suptitle('Score distributions per cell', color='white', fontsize=13)

for ax, col in zip(axes, metric_cols):
    data = [per_image_df[per_image_df['cell'] == c][col].dropna().tolist() for c in CELLS]
    bp = ax.boxplot(
        data,
        patch_artist=True,
        medianprops=dict(color='white', linewidth=2),
        whiskerprops=dict(color='#aaa'),
        capprops=dict(color='#aaa'),
        flierprops=dict(marker='o', color='#888', markersize=4),
    )
    for patch, cell in zip(bp['boxes'], CELLS):
        patch.set_facecolor(CELL_COLORS[cell])
        patch.set_alpha(0.85)

    ax.set_xticklabels([CELL_LABELS[c] for c in CELLS], fontsize=8, rotation=20, ha='right')
    ax.set_title(col.replace('_', ' '), color='white', fontsize=9)
    ax.grid(axis='y', linewidth=0.5, alpha=0.6)

plt.tight_layout()
boxplot_path = EVAL_DIR / 'metrics_box_plots.png'
plt.savefig(boxplot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Box plots saved → {boxplot_path}')

## 8 — Top Success & Worst Failure Grid

Best = highest CLIPScore per cell | Worst = lowest CLIPScore per cell

In [ ]:
fig, axes = plt.subplots(2, len(CELLS), figsize=(len(CELLS) * 3.5, 7))
fig.patch.set_facecolor('#0d0d0d')
fig.suptitle('Best (↑ CLIPScore) and Worst (↓ CLIPScore) per Cell', color='white', fontsize=12)

axes[0][0].set_ylabel('Best', color='#4cba88', fontsize=10, rotation=0, ha='right', labelpad=45)
axes[1][0].set_ylabel('Worst', color='#e05555', fontsize=10, rotation=0, ha='right', labelpad=45)

for col_i, cell in enumerate(CELLS):
    cell_data = per_image_df[per_image_df['cell'] == cell]
    if 'clip_score' not in cell_data.columns or cell_data.empty:
        for row_i in range(2):
            axes[row_i][col_i].axis('off')
        continue

    sorted_data = cell_data.sort_values('clip_score')
    best_row    = sorted_data.iloc[-1]
    worst_row   = sorted_data.iloc[0]

    for row_i, (row, border_color) in enumerate([(best_row, '#4cba88'), (worst_row, '#e05555')]):
        ax = axes[row_i][col_i]
        stem    = row['stem']
        img_path = OUT_ROOT / cell / f'{stem}.png'
        if img_path.exists():
            ax.imshow(Image.open(img_path))
        else:
            ax.imshow(Image.new('RGB', (512, 512), (40, 40, 40)))
        ax.axis('off')
        score = row.get('clip_score', float('nan'))
        subtitle = f'{cell}: {row.get("breed","").replace("_", " ")}\n{row.get("condition","")}\nCLIP={score:.3f}'

        if row_i == 0:
            ax.set_title(subtitle, color=border_color, fontsize=7, pad=4)
        else:
            ax.set_xlabel(subtitle, color=border_color, fontsize=7, labelpad=4)

        for spine in ax.spines.values():
            spine.set_edgecolor(border_color)
            spine.set_linewidth(2)

plt.tight_layout(pad=0.8)
best_worst_path = EVAL_DIR / 'best_worst_grid.png'
plt.savefig(best_worst_path, dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print(f'Best/worst grid saved → {best_worst_path}')

## 9 — Condition-Level Breakdown

CLIPScore by condition across cells — shows which veterinary conditions are easiest/hardest to generate.

In [ ]:
if 'clip_score' in per_image_df.columns:
    cond_df = (
        per_image_df.groupby(['condition', 'cell'])['clip_score']
        .mean()
        .unstack('cell')
        .sort_values('D' if 'D' in per_image_df['cell'].unique() else 'A', ascending=False)
    )

    conditions = cond_df.index.tolist()
    x = np.arange(len(conditions))
    bar_w = 0.18

    fig, ax = plt.subplots(figsize=(14, 5))
    fig.patch.set_facecolor('#0d0d0d')

    for i, cell in enumerate(CELLS):
        if cell not in cond_df.columns:
            continue
        offset = (i - len(CELLS) / 2 + 0.5) * bar_w
        ax.bar(x + offset, cond_df[cell], width=bar_w,
               color=CELL_COLORS[cell], label=CELL_LABELS[cell],
               edgecolor='#444', linewidth=0.6)

    ax.set_xticks(x)
    ax.set_xticklabels([c.replace('_', ' ') for c in conditions],
                       rotation=30, ha='right', fontsize=9)
    ax.set_ylabel('Mean CLIPScore', fontsize=10)
    ax.set_title('CLIPScore by Veterinary Condition & Cell', color='white', fontsize=12)
    ax.legend(fontsize=8, facecolor='#1a1a1a', edgecolor='#555', labelcolor='white')
    ax.grid(axis='y', linewidth=0.5, alpha=0.6)

    plt.tight_layout()
    cond_chart_path = EVAL_DIR / 'condition_breakdown.png'
    plt.savefig(cond_chart_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Condition breakdown saved → {cond_chart_path}')
else:
    print('clip_score column not found — skipping condition breakdown.')

## 10 — Read the Generated Report.md

In [ ]:
from IPython.display import Markdown, display

report_path = EVAL_DIR / 'report.md'
if report_path.exists():
    display(Markdown(report_path.read_text()))
else:
    print('report.md not found — run build_report() above first.')

## 11 — Findings Write-Up

Fill this in after reviewing the charts above. These go directly into the slide deck (slide 9 — Results).

---

### Summary of quantitative findings

| Metric | Expected winner | Actual winner | Margin |
|--------|----------------|--------------|--------|
| CLIPScore | D | _fill_ | _fill_ |
| Breed Identity | D | _fill_ | _fill_ |
| Condition Consistency | D | _fill_ | _fill_ |
| LPIPS Diversity | ≈ equal | _fill_ | _fill_ |

### Does structured prompting alone help? (C vs A)
_e.g. "Cell C outperforms A by +0.023 CLIPScore (+8%), confirming that slot-filled prompts with environment/style anchoring improve alignment even without ControlNet."_

### Does ControlNet alone help? (B vs A)
_e.g. "Cell B shows +0.015 breed identity vs A — shape conditioning anchors the silhouette, making the breed more recognisable even with a naive prompt."_

### Does structured + ControlNet dominate? (D vs rest)
_e.g. "Cell D leads on CLIPScore and breed identity. Condition consistency is also highest, suggesting the combination of detailed prompts + shape control reduces semantic drift."_

### Hardest conditions to generate well
_List 2–3 from the condition breakdown chart_

### Failure cases and hypotheses
_e.g. "Dental check generates plausible images but the tooth-examination detail is rarely depicted correctly. Likely cause: rare in SD 1.5 training data. Possible fix: ControlNet edge conditioning + more specific prompt engineering."_

### LPIPS diversity — mode collapse check
_Confirm all cells > 0.30. If any cell is < 0.30, flag it and hypothesise why._

---
**End of Notebook 04** — Phase 5 complete.  
Next → `src/app/gradio_app.py` (Phase 6 — Gradio demo).